[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/01_grad_basics_solution.ipynb)

# 🟢 Solution: SGD Step with value_and_grad

*JAX Fundamentals · Easy*

Reference implementation. Try it yourself in `01_grad_basics.ipynb` first.

---
Implement a single **stochastic gradient descent step**.

This is the innermost loop of essentially every JAX training script — if you
only learn one JAX idiom, learn this one.

Given a loss function, a pytree of parameters, a batch, and a learning rate,
return the updated parameters and the loss *at the old parameters*.

$$\theta_{t+1} = \theta_t - \eta \nabla_\theta \mathcal{L}(\theta_t, \text{batch})$$

### Rules
- Use `jax.value_and_grad` — one forward and one backward pass give you both
  numbers (`loss_fn(...)` followed by `jax.grad(loss_fn)(...)` runs the forward
  computation twice)
- `params` is an arbitrary **pytree** (nested dicts/lists/tuples of arrays),
  not a flat array — do not assume it is a single array
- Differentiate with respect to `params` only, not the batch
- The returned loss must be the loss **before** the update

### The traps
- **Reporting the loss after the update.** Re-evaluating `loss_fn(new_params, …)`
  costs an extra forward pass *and* reports a number that no training curve
  in the literature plots. `value_and_grad` gives you the value at the point
  where the gradient was taken, which is what you want.
- **Assuming a flat array.** `params - lr * grads` is fine for the toy case and
  dies with `TypeError: unsupported operand type(s) for -: 'dict' and 'float'`
  the moment `params` is a real parameter tree.
- **Reaching for in-place updates.** JAX arrays are immutable; there is no
  `p -= lr * g`. The step *returns* new parameters, which is why JAX training
  loops thread state through explicitly instead of hiding it inside a mutable
  optimizer object.

### Signature
```python
def sgd_step(loss_fn, params, batch, lr):
    # loss_fn(params, batch) -> scalar
    # returns (new_params, loss)
    ...
```

### Why it matters
Production code uses Optax, and `optax.apply_updates` is exactly the `jax.tree`
traversal you are writing here (it adds already-negated updates leaf by leaf).
Interviewers ask for the hand-rolled version to check you understand the
contract underneath it: gradients mirror the parameter tree leaf for leaf, and
the step is a pure function of `(params, batch)` — nothing is mutated — so the
whole thing can be wrapped in `jax.jit` once and reused unchanged.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax


def sgd_step(loss_fn, params, batch, lr):
    loss, grads = jax.value_and_grad(loss_fn)(params, batch)
    new_params = jax.tree.map(lambda p, g: p - lr * g, params, grads)
    return new_params, loss

In [ ]:
# 🔍 Verify
import jax.numpy as jnp

params = {"w": jnp.array([1.0, 2.0]), "b": jnp.array(0.5)}
batch = (jnp.array([[1.0, 0.0], [0.0, 1.0]]), jnp.array([1.0, 1.0]))


def loss_fn(p, b):
    x, y = b
    pred = x @ p["w"] + p["b"]
    return jnp.mean((pred - y) ** 2)


new_params, loss = sgd_step(loss_fn, params, batch, lr=0.1)
print("loss before step:", loss)
print("old params:", params)
print("new params:", new_params)

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("grad_basics")